In [1]:
import numpy as np
import pandas as pd
import scipy.io as sio
import nibabel as nib
from pathlib import Path
from nilearn.glm.first_level import compute_regressor

# ============================================================================
# 1. CONFIGURATION & PATHS
# ============================================================================
ROOT = Path("/home/amin/Research/Representation/Movie/data/Setareh")
RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/sitmulus_regressor_cifti")

TIMING_CSV = ROOT / "HCP Data" / "HCP_7T_Movie_Clip_Timing.csv" 
MAT_FILE_LEFT = ROOT / "HCP Data/notmean_left_Meanfile.mat"
MAT_FILE_RIGHT = ROOT / "HCP Data/notmean_right_Meanfile.mat"
TEMPLATE_CIFTI = '/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii'

TOTAL_TRS = 3655
TR_DUR = 1.0

# HCP 7T Movie standard run lengths (summing to 3655 TRs)
RUN_TRS = [921, 918, 915, 901]
RUN_OFFSETS = np.cumsum([0] + RUN_TRS[:-1])

# ============================================================================
# 2. LOAD & FORMAT TIMING DATA 
# ============================================================================
print("Parsing stimulus timing...")
df = pd.read_csv(TIMING_CSV, header=1)
df.columns = ['movie_block', 'start_time', 'end_time', 'duration', 'comment']

# Filter out rest blocks AND drop any rows with NaN in the timing columns
stim_blocks = df[~df['movie_block'].astype(str).isin(['0', '0.0', 'nan', 'NaN'])].copy()
stim_blocks = stim_blocks.dropna(subset=['start_time', 'duration'])

onsets_abs = stim_blocks['start_time'].values
durations = stim_blocks['duration'].values
amplitudes = np.ones(len(onsets_abs))

print(f"Found {len(onsets_abs)} valid stimulus blocks.")

# Group events by run based on absolute timing, to prevent HRF bleed-over
run_events = [[], [], [], []]
for onset, duration, amp in zip(onsets_abs, durations, amplitudes):
    # Determine which run this onset belongs to
    run_idx = 0
    for i, offset in enumerate(RUN_OFFSETS):
        if onset >= offset:
            run_idx = i
    
    # Calculate relative onset within the run for compute_regressor
    rel_onset = onset - RUN_OFFSETS[run_idx]
    run_events[run_idx].append([rel_onset, duration, amp])

# ============================================================================
# 3. LOAD BOLD DATA & COMPUTE MASKS
# ============================================================================
print("Loading fMRI arrays from .mat files...")

def safe_load_and_mask(mat_path, data_key):
    mat_data = sio.loadmat(mat_path)
    fmri_raw = mat_data[data_key]
    if fmri_raw.shape[0] != 32492: 
        fmri_raw = fmri_raw.T
    # Create a mask using standard deviation to ignore dead space
    valid_mask = fmri_raw.std(axis=1) > 0
    return fmri_raw, valid_mask

fmri_left_raw, valid_mask_left = safe_load_and_mask(MAT_FILE_LEFT, 'Left_data')
fmri_right_raw, valid_mask_right = safe_load_and_mask(MAT_FILE_RIGHT, 'Right_data')

template_img = nib.load(TEMPLATE_CIFTI)

# ============================================================================
# 4. PER-RUN fMRI PROCESSING FUNCTION
# ============================================================================
def process_fmri_per_run(fmri_raw, valid_mask, is_normalized):
    """
    Processes the concatenated fMRI array by splitting it into its original runs,
    applying demean/Z-scoring *within* each run, and concatenating the result.
    Applies variance calculation ONLY to valid masked vertices to avoid artifacting.
    """
    processed_runs = []
    current_tr = 0
    
    for n_trs in RUN_TRS:
        run_data = fmri_raw[:, current_tr : current_tr + n_trs]
        
        # 1. Demean per run
        run_mean = np.mean(run_data, axis=1, keepdims=True)
        run_proc = run_data - run_mean
        
        # 2. Z-score per run safely (if normalized)
        if is_normalized:
            run_std = np.ones((run_data.shape[0], 1))
            run_std[valid_mask, 0] = np.std(run_data[valid_mask, :], axis=1) + 1e-8
            run_proc = run_proc / run_std
            
        processed_runs.append(run_proc)
        current_tr += n_trs
        
    return np.hstack(processed_runs)

# ============================================================================
# 5. COMPUTE THE 4 VARIANTS
# ============================================================================
print("\n" + "="*50)
print("STARTING VARIANT GENERATION")
print("="*50)

for use_hrf in [False, True]:
    # --- A. Generate Base Regressor ---
    if use_hrf:
        print("\n--- Model: SPM HRF ---")
        hrf_lbl = "hrf"
        reg_runs = []
        
        # Convolve HRF independently per-run to prevent run-boundary bleed-over
        for run_idx, n_trs in enumerate(RUN_TRS):
            if len(run_events[run_idx]) > 0:
                events_arr = np.array(run_events[run_idx])
                frame_times = np.arange(n_trs) * TR_DUR
                convolved_reg, _ = compute_regressor(events_arr.T, hrf_model='spm', frame_times=frame_times)
                reg_runs.append(convolved_reg.flatten())
            else:
                reg_runs.append(np.zeros(n_trs))
                
        base_regressor = np.concatenate(reg_runs)
        
    else:
        print("\n--- Model: 5s Fixed Delay Boxcar ---")
        hrf_lbl = "5sdelay"
        base_regressor = np.zeros(TOTAL_TRS)
        for onset, duration in zip(onsets_abs, durations):
            start_idx = int(np.round(onset + 5.0))
            end_idx = int(np.round(onset + duration + 5.0))
            
            # Prevent indexing out of bounds
            start_idx = min(max(start_idx, 0), TOTAL_TRS)
            end_idx = min(max(end_idx, 0), TOTAL_TRS)
            base_regressor[start_idx:end_idx] = 1.0

    for is_normalized in [False, True]:
        norm_lbl = "normalized" if is_normalized else "demeaned"
        print(f"  -> Processing: {hrf_lbl}_{norm_lbl}")

        # --- B. Process Regressor ---
        reg_mean = np.mean(base_regressor)
        if is_normalized:
            reg_proc = (base_regressor - reg_mean) / (np.std(base_regressor) + 1e-8)
        else:
            reg_proc = (base_regressor - reg_mean)

        # --- C. Process fMRI & Compute Correlation ---
        left_proc = process_fmri_per_run(fmri_left_raw, valid_mask_left, is_normalized)
        right_proc = process_fmri_per_run(fmri_right_raw, valid_mask_right, is_normalized)

        # Ensure the regressor is also processed (demeaned and scaled by its own STD)
        reg_centered = reg_proc - np.mean(reg_proc)
        reg_final = reg_centered / (np.std(reg_centered) + 1e-8)

        def get_correlation(data_array, regressor):
            # 1. Mean-center both to be absolutely safe
            data_cent = data_array - np.mean(data_array, axis=1, keepdims=True)
            reg_cent = regressor - np.mean(regressor)
            
            # 2. Covariance (Numerator)
            cov = np.dot(data_cent, reg_cent)
            
            # 3. Product of Sum of Squares (Denominator)
            ss_data = np.sum(data_cent**2, axis=1)
            ss_reg = np.sum(reg_cent**2)
            
            denominator = np.sqrt(ss_data * ss_reg) + 1e-8
            return cov / denominator

        left_corr = get_correlation(left_proc, base_regressor) 
        right_corr = get_correlation(right_proc, base_regressor)

        # --- D. Mask and Save CIFTI ---
        left_clean = np.nan_to_num(left_corr, nan=0.0)[valid_mask_left]
        right_clean = np.nan_to_num(right_corr, nan=0.0)[valid_mask_right]

        combined_data = np.hstack((left_clean, right_clean)).reshape(1, -1)

        if combined_data.shape[1] != template_img.shape[1]:
            raise ValueError(
                f"Shape mismatch! Concatenated data: {combined_data.shape[1]} vertices, "
                f"Template: {template_img.shape[1]}."
            )

        cifti = nib.Cifti2Image(
            combined_data.astype(np.float32), 
            header=template_img.header,
            nifti_header=template_img.nifti_header
        )

        try:
            scalar_axis = cifti.header.get_axis(0)
            scalar_axis.name[0] = f"Stimulus_Correlation_{hrf_lbl}_{norm_lbl}"
        except Exception as e:
            pass

        cifti_dir = RESULTS_BASE 
        cifti_dir.mkdir(parents=True, exist_ok=True)
        
        map_output = cifti_dir / f"HCP_movie_stimulus_correlation_{hrf_lbl}_{norm_lbl}.dscalar.nii"
        nib.save(cifti, map_output)
        print(f"  [SUCCESS] Saved CIFTI map: {map_output.name}")

print("\nAll 4 maps generated successfully.")

Parsing stimulus timing...
Found 18 valid stimulus blocks.
Loading fMRI arrays from .mat files...

STARTING VARIANT GENERATION

--- Model: 5s Fixed Delay Boxcar ---
  -> Processing: 5sdelay_demeaned
  [SUCCESS] Saved CIFTI map: HCP_movie_stimulus_correlation_5sdelay_demeaned.dscalar.nii
  -> Processing: 5sdelay_normalized
  [SUCCESS] Saved CIFTI map: HCP_movie_stimulus_correlation_5sdelay_normalized.dscalar.nii

--- Model: SPM HRF ---
  -> Processing: hrf_demeaned
  [SUCCESS] Saved CIFTI map: HCP_movie_stimulus_correlation_hrf_demeaned.dscalar.nii
  -> Processing: hrf_normalized
  [SUCCESS] Saved CIFTI map: HCP_movie_stimulus_correlation_hrf_normalized.dscalar.nii

All 4 maps generated successfully.
